# Screening Decisions Analysis

Analyze the paper screening decisions made by each rater and export the final included papers as a BibTeX file.

In [22]:
import re
from pathlib import Path
import os

import bibtexparser
import pandas as pd

from sklearn.metrics import cohen_kappa_score

project_root = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").exists()
)
library_path = project_root / "papers" / "papers.bib"
included_bib_path = project_root / "artifacts" / "included.bib"

## Screening Decisions

Load each rater's decisions and inspect the include/exclude distribution.

In [4]:
lucas_df = pd.read_csv("../decisions/lucas.csv", usecols=["id", "decision", "reason"])
lucas_df.head()

,id,decision,reason
0,52,include,IC1
1,60,include,IC2
2,71,include,IC2
3,77,include,IC1
4,107,exclude,EC2


In [5]:
lucas_df["decision"].value_counts()

decision
exclude    211
include     33
Name: count, dtype: int64

In [34]:
victor_df = pd.read_csv("../decisions/victor.csv", usecols=["id", "decision", "reason", "notes"], dtype={"notes": str})
victor_df.head()

,id,decision,reason,notes
0,52,exclude,EC2,NaN
1,60,include,IC1,NaN
2,71,include,IC2,NaN
3,77,include,IC1,NaN
4,107,exclude,EC2,NaN


In [35]:
victor_df["decision"].value_counts()

decision
exclude    160
include     84
Name: count, dtype: int64

In [36]:
kappa = cohen_kappa_score(lucas_df["decision"], victor_df["decision"])
print(f"Cohen's Kappa: {kappa:.3f}")

Cohen's Kappa: 0.374


> What magnitude of kappa reflects adequate agreement?

There are many guidelines that tries to answer this question, but any set of guidelines is however by no means universally accepted (usually arbitrary and based on personal opinion).

Anyways, Fleiss's guidelines characterize kappas over 0.75 as excellent, 0.40 to 0.75 as fair to good, and below 0.40 as poor. We can say that our agreement is almost **fair**.

In [44]:
all_papers_df = pd.read_csv("../artifacts/all_papers.csv", usecols=["canonical_id","title","abstract"])
all_papers_df.head()

,canonical_id,title,abstract
0,1,Sustainability and\&nbsp;High Performance Comp...,The use of High Performance Computing (HPC) ha...
1,2,Environment-conscious scheduling of HPC applic...,The use of High Performance Computing (HPC) in...
2,3,A Digital Twin Framework for Liquid-cooled Sup...,"We present ExaDigiT, an open-source framework ..."
3,4,Toward Sustainable HPC: In-Production Deployme...,This paper describes the deployment and operat...
4,5,Designing an Energy-Efficient HPC Supercomputi...,This paper presents design considerations that...


In [48]:
conflicts_df = lucas_df[lucas_df["decision"] != victor_df["decision"]].copy()
conflicts_df.rename(columns={"decision": "lucas_decision", "reason": "lucas_reason"}, inplace=True)

conflicts_df["victor_decision"] = victor_df.loc[conflicts_df.index, "decision"]
conflicts_df["victor_reason"] = victor_df.loc[conflicts_df.index, "reason"]
conflicts_df["victor_notes"] = victor_df.loc[conflicts_df.index, "notes"]

conflicts_df = pd.merge(conflicts_df, all_papers_df, left_on="id", right_on="canonical_id")

conflicts_df = conflicts_df[["id", "title", "abstract", "lucas_decision", "lucas_reason", "victor_decision", "victor_reason", "victor_notes"]]

conflicts_df["final_decision"] = ""
conflicts_df["resolution"] = ""
conflicts_df.head()

,id,title,abstract,lucas_decision,lucas_reason,victor_decision,victor_reason,victor_notes,final_decision,resolution
0,52,The Power of\&nbsp;Training: How Different Neu...,This work offers a heuristic evaluation of the...,include,IC1,exclude,EC2,NaN,,
1,223,Data Center Operators Face Energy Irony,High-performance computational technology is e...,exclude,EC1,include,IC1,NaN,,
2,237,"Sustainable AI: Experiences, Challenges \&amp;...",The use of Artificial Intelligence (AI) and Ma...,exclude,EC2,include,IC1,NaN,,
3,408,Empowering Generative AI in Enterprises: Susta...,Rapid growth in unstructured data has triggere...,exclude,EC2,include,IC1,NaN,,
4,445,A deep dive into sustainable generative AI and...,The lecture covers the foundations of sustaina...,exclude,EC1,include,IC1,NaN,,


In [49]:
conflicts_csv_path = "../artifacts/conflicts.csv"

if not os.path.exists(conflicts_csv_path):
    print(f"Exporting conflicts to {conflicts_csv_path}...")
    conflicts_df.to_csv(conflicts_csv_path, index=False)
else:
    print(f"Conflicts file already exists at {conflicts_csv_path}....")

Exporting conflicts to ../artifacts/conflicts.csv...


### Merging decisions

In [ ]:
first_df = pd.read_csv(project_root / "first.csv")
first_df.head()

,id,decision,reason
0,1,include,IC1
1,2,include,IC1
2,3,include,IC1
3,4,include,IC1
4,5,include,IC1


In [25]:
all_decisions = pd.concat([first_df, lucas_df])
all_decisions["decision"].value_counts()

decision
exclude    545
include     64
Name: count, dtype: int64

In [14]:
to_include_df = lucas_df[lucas_df["decision"] == "include"]
to_include_df.head()

,id,decision,reason
0,52,include,IC1
1,60,include,IC2
2,71,include,IC2
3,77,include,IC1
13,188,include,IC1


## BibTeX Export

Load the full BibTeX library and `artifacts/all_papers.csv`, match each included paper by title, and write the results to `artifacts/included.bib`.

In [ ]:
with library_path.open(encoding="utf-8") as f:
    library = bibtexparser.parse_string(f.read())

In [ ]:
papers_df = pd.read_csv(project_root / "artifacts" / "all_papers.csv", usecols=["id", "canonical_id", "title", "abstract"])
included_papers = papers_df[papers_df["id"].isin(to_include_df["id"])]
print(f"Papers to include: {len(included_papers)}")
included_papers.head()

Papers to include: 34


,id,canonical_id,title,abstract
51,52,52,The Power of\&nbsp;Training: How Different Neu...,This work offers a heuristic evaluation of the...
59,60,60,The Energy Efficiency Research of\&nbsp;Code f...,Last ten years the top performance of the fast...
70,71,71,What A Waste,The immense demand for high performance comput...
76,77,77,Adaptive Carbon-Aware Scheduling Policies for\...,In response to growing energy costs and carbon...
187,188,188,Enabling distributed generation powered sustai...,The necessity for capping carbon emission has ...


In [21]:
import re

def normalize_title(t):
    # Collapse \cmd{arg} -> \cmdarg so CSV and bib titles compare equal
    return re.sub(r'\{(\w+)\}', r'\1', t).strip()

title_to_entry = {
    normalize_title(e.fields_dict["title"].value): e
    for e in library.entries
    if "title" in e.fields_dict
}

included_entries = []
missing = []
for _, row in included_papers.iterrows():
    entry = title_to_entry.get(normalize_title(row["title"]))
    if entry:
        included_entries.append(entry)
    else:
        missing.append(row["id"])

print(f"Found: {len(included_entries)}, Missing: {len(missing)}")
if missing:
    print("Missing IDs:", missing)

Found: 34, Missing: 0


In [ ]:
out_lib = bibtexparser.Library()
for entry in included_entries:
    out_lib.add(entry)

bibtex_str = bibtexparser.write_string(out_lib)
with included_bib_path.open("w", encoding="utf-8") as f:
    f.write(bibtex_str)

print(f"Written {len(included_entries)} entries to {included_bib_path.relative_to(project_root)}")

Written 34 entries to artifacts/included.bib
